<a href="https://colab.research.google.com/github/Lucky-m-code/Abamela/blob/main/char_rnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install torch requests

In [3]:
import os
import requests
import torch
import torch.nn as nn
import numpy as np

In [4]:
# =========================
# 1. Device Configuration
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


In [5]:
# =========================
# 2. Download Dataset Automatically
# =========================
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
filename = "shakespeare.txt"

if not os.path.exists(filename):
    print("Downloading Shakespeare dataset...")

    response = requests.get(url)

    with open(filename, "w", encoding="utf-8") as f:
        f.write(response.text)

    print("Dataset downloaded successfully!")

Dataset downloaded successfully!


In [6]:
# =========================
# 3. Load Dataset
# =========================
with open(filename, "r", encoding="utf-8") as f:
    text = f.read()

print("Dataset length:", len(text))

Dataset length: 1115394


In [7]:
# =========================
# 4. Character Encoding
# =========================
chars = sorted(list(set(text)))

vocab_size = len(chars)

print("Unique characters:", vocab_size)

char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for ch, i in char_to_idx.items()}

# Encode entire dataset
encoded_text = np.array([char_to_idx[c] for c in text])

Unique characters: 65


In [8]:
# =========================
# 5. Hyperparameters
# =========================
seq_length = 100
hidden_size = 256
num_layers = 2
batch_size = 64
learning_rate = 0.001
epochs = 10

# =========================
# 6. Create Input Sequences
# =========================
def create_sequences(data, seq_length):
    inputs = []
    targets = []

    for i in range(0, len(data) - seq_length):
        inputs.append(data[i:i + seq_length])
        targets.append(data[i + 1:i + seq_length + 1])

    return np.array(inputs), np.array(targets)

X, Y = create_sequences(encoded_text, seq_length)

print("Total sequences:", len(X))

Total sequences: 1115294


In [9]:
# =========================
# 7. Convert to Tensors
# =========================
X = torch.tensor(X, dtype=torch.long)
Y = torch.tensor(Y, dtype=torch.long)

dataset = torch.utils.data.TensorDataset(X, Y)

loader = torch.utils.data.DataLoader(
    dataset,
    batch_size=batch_size,
    shuffle=True
)

In [10]:
# =========================
# 8. Define Character-Level RNN
# =========================
class CharRNN(nn.Module):

    def __init__(self, vocab_size, hidden_size, num_layers):
        super(CharRNN, self).__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # Character embeddings
        self.embedding = nn.Embedding(vocab_size, hidden_size)

        # LSTM
        self.lstm = nn.LSTM(
            hidden_size,
            hidden_size,
            num_layers,
            batch_first=True
        )

        # Output layer
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden):

        x = self.embedding(x)

        out, hidden = self.lstm(x, hidden)

        out = self.fc(out)

        return out, hidden

    def init_hidden(self, batch_size):

        h0 = torch.zeros(
            self.num_layers,
            batch_size,
            self.hidden_size
        ).to(device)

        c0 = torch.zeros(
            self.num_layers,
            batch_size,
            self.hidden_size
        ).to(device)

        return (h0, c0)

In [11]:
# =========================
# 9. Initialize Model
# =========================
model = CharRNN(
    vocab_size,
    hidden_size,
    num_layers
).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=learning_rate
)

In [ ]:
# =========================
# 10. Training Loop
# =========================
print("\nStarting Training...\n")

for epoch in range(epochs):

    total_loss = 0

    for inputs, targets in loader:

        inputs = inputs.to(device)
        targets = targets.to(device)

        hidden = model.init_hidden(inputs.size(0))

        # Detach hidden state
        hidden = tuple([h.detach() for h in hidden])

        # Forward pass
        outputs, hidden = model(inputs, hidden)

        # Reshape for loss calculation
        loss = criterion(
            outputs.view(-1, vocab_size),
            targets.view(-1)
        )

        # Backpropagation
        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(loader)

    print(f"Epoch [{epoch+1}/{epochs}] Loss: {avg_loss:.4f}")


Starting Training...

Epoch [1/10] Loss: 1.1193
Epoch [2/10] Loss: 0.7857
Epoch [3/10] Loss: 0.6847
Epoch [4/10] Loss: 0.6401
Epoch [5/10] Loss: 0.6138
Epoch [6/10] Loss: 0.5963
Epoch [7/10] Loss: 0.5834
Epoch [8/10] Loss: 0.5734
Epoch [9/10] Loss: 0.5653
Epoch [10/10] Loss: 0.5588


In [ ]:
# =========================
# 11. Text Generation Function
# =========================
def generate_text(model, start_text, length=300, temperature=1.0):

    model.eval()

    generated_text = start_text

    # Convert starting text to tensor
    input_seq = torch.tensor(
        [char_to_idx[c] for c in start_text],
        dtype=torch.long
    ).unsqueeze(0).to(device)

    hidden = model.init_hidden(1)

    for _ in range(length):

        output, hidden = model(input_seq, hidden)

        # Temperature scaling
        logits = output[0, -1] / temperature

        probs = torch.softmax(logits, dim=0).detach().cpu().numpy()

        # Sample next character
        next_char_idx = np.random.choice(
            len(probs),
            p=probs
        )

        next_char = idx_to_char[next_char_idx]

        generated_text += next_char

        # Prepare next input
        input_seq = torch.tensor(
            [[next_char_idx]],
            dtype=torch.long
        ).to(device)

    return generated_text


In [ ]:
# =========================
# 12. Generate Text Samples
# =========================
print("\n==============================")
print("LOW TEMPERATURE SAMPLE (0.5)")
print("==============================\n")

print(generate_text(
    model,
    "To be or not to be ",
    temperature=0.5
))

print("\n==============================")
print("HIGH TEMPERATURE SAMPLE (1.2)")
print("==============================\n")

print(generate_text(
    model,
    "To be or not to be ",
    temperature=1.2
))



LOW TEMPERATURE SAMPLE (0.5)

To be or not to be so doubt.

JULIET:
Chick them that he is dead, that she is but a wife
To curb you have and such a mystery:
but what my colours of supper;
And I, that can as mine have to the elder,
The priest let me tell the time to except.
I tell thee, holding up the nurse, and pry on thee,
I have with her two cous

HIGH TEMPERATURE SAMPLE (1.2)

To be or not to be husbanded;
For that he is in thy dying 'Is thanks and
More tongue can find one time to breathe.

DUKE VINCENTIO:
Have you got up the lark, the king is loved to worn
The silvant Warwick served of Norfolk:
And bowly: my grave is gnatched and people: if thou
Hast comfort, for the King of Norfolk,
That 


In [ ]:
# =========================
# 13. Simple Non-Coder Demo
# =========================

def simple_demo():

    print("\n" + "="*50)
    print(" SIMPLE VISUAL DEMO: HOW THE MODEL WORKS")
    print("="*50)

    print("\nSTEP 1: Enter Your Own Starting Text")

    seed_text = input("Type a starting sentence: ")

    # Default fallback
    if seed_text.strip() == "":
        seed_text = "To be or not to be "

    print("\nYou entered:")
    print(seed_text)

    print("\nSTEP 2: Character-Level Learning")
    print("The model reads text character by character:")

    for ch in seed_text[:15]:
        print(ch, end=" → ")

    print("...")

    print("\n\nSTEP 3: Model Architecture")
    print("Input Characters → Embedding → LSTM Memory → Output Character")

    print("\nSTEP 4: Temperature Comparison")
    print("Low temperature = safer and more predictable")
    print("High temperature = more creative but less stable")

    print("\n" + "="*50)
    print(" LOW TEMPERATURE OUTPUT (0.5)")
    print("="*50)

    low_output = generate_text(
        model,
        seed_text,
        length=250,
        temperature=0.5
    )

    print(low_output)

    print("\n" + "="*50)
    print(" HIGH TEMPERATURE OUTPUT (1.2)")
    print("="*50)

    high_output = generate_text(
        model,
        seed_text,
        length=250,
        temperature=1.2
    )

    print(high_output)

    print("\nSTEP 5: Simple Explanation")
    print("The model does not truly understand Shakespeare.")
    print("It learns character patterns from the dataset")
    print("and predicts what character should come next.")

    print("\nDEMO FINISHED")


# Run the demo
simple_demo()


 SIMPLE VISUAL DEMO: HOW THE MODEL WORKS

STEP 1: Enter Your Own Starting Text
Type a starting sentence: freedo

You entered:
freedo

STEP 2: Character-Level Learning
The model reads text character by character:
f → r → e → e → d → o → ...


STEP 3: Model Architecture
Input Characters → Embedding → LSTM Memory → Output Character

STEP 4: Temperature Comparison
Low temperature = safer and more predictable
High temperature = more creative but less stable

 LOW TEMPERATURE OUTPUT (0.5)


NameError: name 'generate_text' is not defined

In [ ]:
# =========================
# 14. Save Model
# =========================
torch.save(model.state_dict(), "char_rnn_model.pth")

print("\nModel saved successfully!")